# Data Engineering for Final Datasets

This notebook adds engineered features (same logic as `join_datasets.ipynb`) to the six Final Datasets in `Datasets_Ours/Final Datasets`.

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

# Resolve project root robustly
cwd = os.getcwd()
cwd_base = os.path.basename(cwd)
if cwd_base == "preprocessing":
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..", ".."))
elif cwd_base == "Notebooks_Ours":
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
else:
    PROJECT_ROOT = os.path.abspath(cwd)

FINAL_DIR = Path(PROJECT_ROOT) / "Datasets_Ours" / "Final Datasets"
FILES = [
    "drp_training_complete.csv",
    "ec_training_complete.csv",
    "ta_training_complete.csv",
    "drp_validation.csv",
    "ec_validation.csv",
    "ta_validation.csv",
]

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    engineered = df.copy()
    eps = 1e-6

    # --- Date cyclic features ---
    if "sample_date" in engineered.columns:
        dates = pd.to_datetime(engineered["sample_date"], dayfirst=True, errors="coerce")
        months = dates.dt.month.fillna(1).astype(int)
        engineered["month_sin"] = np.sin(2 * np.pi * months / 12)
        engineered["month_cos"] = np.cos(2 * np.pi * months / 12)

    # --- Landsat features ---
    if "landsat_present" not in engineered.columns:
        landsat_cols = [c for c in ["nir", "ndmi", "blue", "red", "swir22", "swir16", "green", "mndwi"] if c in engineered.columns]
        if landsat_cols:
            engineered["landsat_present"] = (~engineered[landsat_cols].isna().any(axis=1)).astype(int)

    if all(c in engineered.columns for c in ["nir", "red"]):
        engineered["ndvi"] = (engineered["nir"] - engineered["red"]) / (engineered["nir"] + engineered["red"] + eps)

    if all(c in engineered.columns for c in ["nir", "swir22"]):
        engineered["nbr"] = (engineered["nir"] - engineered["swir22"]) / (engineered["nir"] + engineered["swir22"] + eps)

    if all(c in engineered.columns for c in ["swir16", "swir22"]):
        engineered["mineral_index"] = (engineered["swir16"] - engineered["swir22"]) / (engineered["swir16"] + engineered["swir22"] + eps)
        engineered["swir_ratio"] = engineered["swir16"] / (engineered["swir22"] + eps)

    if all(c in engineered.columns for c in ["swir22", "nir"]):
        engineered["salinity_proxy"] = engineered["swir22"] / (engineered["nir"] + eps)

    # --- TerraClimate features ---
    if all(c in engineered.columns for c in ["ppt", "pet"]):
        engineered["wb"] = engineered["ppt"] - engineered["pet"]
        engineered["eci"] = engineered["pet"] / (engineered["ppt"] + eps)

    if all(c in engineered.columns for c in ["q", "ppt"]):
        engineered["rr"] = engineered["q"] / (engineered["ppt"] + eps)

    if all(c in engineered.columns for c in ["pet", "aet"]):
        engineered["etgap"] = engineered["pet"] - engineered["aet"]

    if all(c in engineered.columns for c in ["def", "ppt"]):
        engineered["dsi"] = engineered["def"] / (engineered["ppt"] + eps)

    if all(c in engineered.columns for c in ["srad", "tmax", "tmin"]):
        engineered["hri"] = engineered["srad"] * ((engineered["tmax"] + engineered["tmin"]) / 2)

    # --- JRC buffer features (if present) ---
    if all(c in engineered.columns for c in ["gsw_occurrence_mean_1km", "gsw_seasonality_mean_1km"]):
        engineered["water_perm"] = engineered["gsw_occurrence_mean_1km"] * (engineered["gsw_seasonality_mean_1km"] / 12)
        engineered["water_instab"] = (1 - engineered["gsw_occurrence_mean_1km"] / 100) * (engineered["gsw_seasonality_mean_1km"] / 12)

    if all(c in engineered.columns for c in ["gsw_recurrence_mean_1km", "gsw_extent_mean_1km"]):
        engineered["recurrence_ratio"] = engineered["gsw_recurrence_mean_1km"] / (engineered["gsw_extent_mean_1km"] + eps)

    if "gsw_seasonality_mean_1km" in engineered.columns:
        engineered["seasonal_water"] = engineered["gsw_seasonality_mean_1km"].between(1, 9).astype(int)

    # --- ESA point-based features (if present) ---
    if "esa_change_count" in engineered.columns:
        engineered["esa_change_intensity"] = np.log1p(engineered["esa_change_count"])

    # --- GAIA buffer interactions (if present) ---
    if all(c in engineered.columns for c in ["wb", "gaia_impervious_frac_by_sample_year_1km"]):
        engineered["wb_x_impervious"] = engineered["wb"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

    if all(c in engineered.columns for c in ["eci", "gaia_impervious_frac_by_sample_year_1km"]):
        engineered["eci_x_impervious"] = engineered["eci"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

    if all(c in engineered.columns for c in ["gsw_occurrence_mean_1km", "gaia_impervious_frac_by_sample_year_1km"]):
        engineered["gsw_occ_x_impervious"] = engineered["gsw_occurrence_mean_1km"] * engineered["gaia_impervious_frac_by_sample_year_1km"]

    return engineered

In [3]:
print("Adding engineered features to Final Datasets...")

for fname in FILES:
    path = FINAL_DIR / fname
    df = pd.read_csv(path)
    before_cols = len(df.columns)

    engineered = add_engineered_features(df)
    after_cols = len(engineered.columns)

    engineered.to_csv(path, index=False)

    print(f"\n{fname}")
    print(f"  Columns: {before_cols} -> {after_cols}")
    print(f"  Shape: {engineered.shape}")

print("\n✓ Engineered features added to all six datasets.")

Adding engineered features to Final Datasets...

drp_training_complete.csv
  Columns: 67 -> 67
  Shape: (9319, 67)

ec_training_complete.csv
  Columns: 65 -> 65
  Shape: (9319, 65)

ta_training_complete.csv
  Columns: 65 -> 65
  Shape: (9319, 65)

drp_validation.csv
  Columns: 64 -> 64
  Shape: (200, 64)

ec_validation.csv
  Columns: 64 -> 64
  Shape: (200, 64)

ta_validation.csv
  Columns: 64 -> 64
  Shape: (200, 64)

✓ Engineered features added to all six datasets.


In [4]:
print("\nVerification preview (first 5 columns + engineered columns):")
for fname in FILES:
    df = pd.read_csv(FINAL_DIR / fname)
    new_cols = [c for c in [
        "month_sin", "month_cos", "landsat_present", "ndvi", "nbr", "mineral_index",
        "swir_ratio", "salinity_proxy", "wb", "rr", "etgap", "eci", "dsi", "hri",
        "water_perm", "water_instab", "recurrence_ratio", "seasonal_water",
        "esa_change_intensity", "wb_x_impervious", "eci_x_impervious", "gsw_occ_x_impervious",
    ] if c in df.columns]

    print(f"\n{fname}")
    print("  First 5 columns:", list(df.columns[:5]))
    print("  Engineered present:", new_cols)


Verification preview (first 5 columns + engineered columns):

drp_training_complete.csv
  First 5 columns: ['latitude', 'longitude', 'month_fitted', 'Total Alkalinity', 'Electrical Conductance']
  Engineered present: ['landsat_present', 'wb', 'rr', 'etgap', 'eci', 'dsi', 'hri', 'water_perm', 'water_instab', 'recurrence_ratio', 'seasonal_water', 'esa_change_intensity', 'wb_x_impervious', 'eci_x_impervious', 'gsw_occ_x_impervious']

ec_training_complete.csv
  First 5 columns: ['latitude', 'longitude', 'month_fitted', 'Total Alkalinity', 'Electrical Conductance']
  Engineered present: ['landsat_present', 'wb', 'rr', 'etgap', 'eci', 'dsi', 'hri', 'water_perm', 'water_instab', 'recurrence_ratio', 'seasonal_water', 'esa_change_intensity', 'wb_x_impervious', 'eci_x_impervious', 'gsw_occ_x_impervious']

ta_training_complete.csv
  First 5 columns: ['latitude', 'longitude', 'month_fitted', 'Total Alkalinity', 'Electrical Conductance']
  Engineered present: ['landsat_present', 'wb', 'rr', 'etgap

In [5]:
# Add month_fitted feature using fitted parameters (if missing)
import pickle

params_path = Path(PROJECT_ROOT) / "Notebooks_Ours" / "preprocessing" / "fitted_monthly_params.pkl"
with open(params_path, "rb") as f:
    fitted_params = pickle.load(f)


def sinusoidal(x, A, B, C, D):
    return A * np.sin(B * x + C) + D


def polynomial_3(x, a, b, c, d):
    return a * x**3 + b * x**2 + c * x + d


def add_month_fitted(df: pd.DataFrame, target_key: str) -> pd.DataFrame:
    if "month_fitted" in df.columns:
        return df
    if "sample_date" not in df.columns:
        return df

    df = df.copy()
    df["sample_date"] = pd.to_datetime(df["sample_date"], errors="coerce")
    df["month"] = df["sample_date"].dt.month

    if target_key == "drp":
        params = fitted_params["DRP"]["params"]
        df["month_fitted"] = polynomial_3(df["month"], *params)
    elif target_key == "ec":
        params = fitted_params["EC"]["params"]
        df["month_fitted"] = sinusoidal(df["month"], *params)
    elif target_key == "ta":
        params = fitted_params["TA"]["params"]
        df["month_fitted"] = sinusoidal(df["month"], *params)

    df = df.drop(columns=[c for c in ["month"] if c in df.columns])
    return df


print("\nAdding month_fitted where missing...")
for fname in FILES:
    path = FINAL_DIR / fname
    df = pd.read_csv(path)
    key = "drp" if fname.startswith("drp_") else "ec" if fname.startswith("ec_") else "ta"

    before = "month_fitted" in df.columns
    df = add_month_fitted(df, key)
    after = "month_fitted" in df.columns

    if not before and after:
        df.to_csv(path, index=False)
        print(f"{fname}: month_fitted added")
    elif before:
        print(f"{fname}: month_fitted already present")
    else:
        print(f"{fname}: month_fitted not added (missing sample_date)")


Adding month_fitted where missing...
drp_training_complete.csv: month_fitted already present
ec_training_complete.csv: month_fitted already present
ta_training_complete.csv: month_fitted already present
drp_validation.csv: month_fitted already present
ec_validation.csv: month_fitted already present
ta_validation.csv: month_fitted already present


## Fitted monthly curves (from `fitted_monthly_params.txt`)

TA:
- Model: sinusoidal
- Equation: `y = -13.7422 * sin(0.4554 * x + 0.8642) + 117.4619`
- R²: `0.963565`
- Parameters: `[-13.74223074, 0.45536533, 0.86416839, 117.46193818]`

EC:
- Model: sinusoidal
- Equation: `y = -62.4782 * sin(0.4683 * x + 0.7949) + 478.6538`
- R²: `0.946747`
- Parameters: `[-62.4782329, 0.468307277, 0.794851525, 478.6538]`

DRP:
- Model: polynomial_3
- Equation: `y = 0.049361*x³ + -0.552989*x² + -0.245165*x + 50.4211`
- R²: `0.916941`
- Parameters: `[0.0493613889, -0.552989084, -0.245165177, 50.4210517]`